## This is Prompt Learning - Optimization

# Content
## Define Classifier
## Define Datasets
## Define DSPy.Examples
## Define Optimizers
## Define Evaluation
## Run Evaluation
## Save Prompts

# Define classifier

In [1]:
import dspy #the dspy library for building and optimizing language model prompts
import random #often used for shuffling examples in few-shot prompting
import pandas as pd #pandas for data manipulation and analysis
from datasets import load_dataset # Hugging Face datasets library for loading and working with datasets

class Classification(dspy.Signature): #Define what the model should do as task for optimizing the prompt. 
    """Classify the customer message into one of the intent labels.
    The output should be only the predicted class as a single intent label."""

    customer_message = dspy.InputField(desc="Customer message during customer service interaction") # Input field for the customer's message
    intent_labels = dspy.InputField(desc="Labels that represent customer intent") # Input field for the possible intent labels
    answer = dspy.OutputField(desc="a label best matching customer's intent ") # Output field for the predicted intent label

# Use Ollama local llama model instead of OpenAI
# 1. Initialize your local Ollama model using the modern LM client
lm_ollama = dspy.LM(
    'ollama_chat/llama3.2:3b', 
    api_base='http://localhost:11434', 
    max_tokens=512
) # Initialize the Ollama language model with the specified parameters
dspy.settings.configure(lm=lm_ollama) # Configure dspy to use the Ollama language model for all subsequent operations
#dspy.predict()
cot_predictor = dspy.ChainOfThought(Classification) # Create a Chain of Thought predictor using the Classification signature RAtionales (CoT) approach

21:14:23 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
21:14:23 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


# Parse Atis Dataset

In [2]:
dataset = load_dataset("tuetschek/atis")
dataset.set_format(type="pandas") #define the format of the dataset to be pandas DataFrame for easier manipulation

df_train: pd.DataFrame = dataset["train"][:] #train set as a pandas DataFrame
df_test: pd.DataFrame = dataset["test"][:] #test set as a pandas DataFrame
small_test = df_test.head(100) #take a small subset of the test set for quicker evaluation

## x column: text, y column: intent

In [3]:
df_train.iloc[0]

id                                                        0
intent                                               flight
text      i want to fly from boston at 838 am and arrive...
slots     O O O O O B-fromloc.city_name O B-depart_time....
Name: 0, dtype: object

## prepare labels

In [4]:
labels = df_train["intent"].unique().tolist()
labels_str = "%".join(labels)
labels_str
#"~|"

'flight%flight_time%airfare%aircraft%ground_service%airport%airline%distance%abbreviation%ground_fare%quantity%city%flight_no%capacity%flight+airfare%meal%restriction%airline+flight_no%ground_service+ground_fare%airfare+flight_time%cheapest%aircraft+flight+flight_no'

In [5]:
## run prediction
first_row = df_train.iloc[0]
print(f"customer message: {first_row['text']},real class: {first_row['intent']}")
cot_predictor(customer_message=first_row["text"], intent_labels=labels_str)

customer message: i want to fly from boston at 838 am and arrive in denver at 1110 in the morning,real class: flight


Prediction(
    reasoning='The customer message contains specific details about a flight from Boston to Denver, including departure time, arrival time, airline, and city.',
    answer='flight_time'
)

In [6]:
cot_predictor.save("cot_zero_shot_firstattempt.json")

In [7]:
cot_predictor.history

[{'prompt': None,
  'messages': [{'role': 'system',
    'content': "Your input fields are:\n1. `customer_message` (str): Customer message during customer service interaction\n2. `intent_labels` (str): Labels that represent customer intent\nYour output fields are:\n1. `reasoning` (str): \n2. `answer` (str): a label best matching customer's intent\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## customer_message ## ]]\n{customer_message}\n\n[[ ## intent_labels ## ]]\n{intent_labels}\n\n[[ ## reasoning ## ]]\n{reasoning}\n\n[[ ## answer ## ]]\n{answer}\n\n[[ ## completed ## ]]\nIn adhering to this structure, your objective is: \n        Classify the customer message into one of the intent labels.\n        The output should be only the predicted class as a single intent label."},
   {'role': 'user',
    'content': '[[ ## customer_message ## ]]\ni want to fly from boston at 838 am and arrive in denver at 1110 in the morning\n\n[[ ## 

# Define Examples

In [8]:
import random
import dspy

# 1. Fixed the uppercase "Example" in the type hint
def get_dspy_examples(df, k) -> list[dspy.Example]:
    dspy_examples = []
    for label in labels:
        try:
            label_df = df[df["intent"] == label].sample(n=k)
            for index, row in label_df.iterrows():
                dspy_examples.append(
                    dspy.Example(
                        customer_message=row["text"], 
                        answer=row["intent"], 
                        intent_labels=labels_str
                    ).with_inputs("customer_message", "intent_labels")
                )
        except ValueError:
            # specifically catch sample size errors if a class has < k items
            continue

    return dspy_examples

# Generate your datasets
train_examples = get_dspy_examples(df_train, k=2)
all_test_examples = get_dspy_examples(df_test, k=10)

# 2. Fixed the splitting logic using indices so DSPy Example identities stay intact
print(f"Total test pool: {len(all_test_examples)}")

# Shuffle indices safely
indices = list(range(len(all_test_examples)))
random.shuffle(indices)

split_idx = len(indices) // 2
dev_indices = set(indices[:split_idx])

dev_examples = [all_test_examples[i] for i in dev_indices]
test_examples = [all_test_examples[i] for i in range(len(all_test_examples)) if i not in dev_indices]

print(f"Dev size: {len(dev_examples)} | Test size: {len(test_examples)}")

Total test pool: 90
Dev size: 45 | Test size: 45


In [9]:
train_examples

[Example({'customer_message': 'show me flights from denver to washington dc wednesday', 'answer': 'flight', 'intent_labels': 'flight%flight_time%airfare%aircraft%ground_service%airport%airline%distance%abbreviation%ground_fare%quantity%city%flight_no%capacity%flight+airfare%meal%restriction%airline+flight_no%ground_service+ground_fare%airfare+flight_time%cheapest%aircraft+flight+flight_no'}) (input_keys={'customer_message', 'intent_labels'}),
 Example({'customer_message': 'what flights from atlanta to toronto', 'answer': 'flight', 'intent_labels': 'flight%flight_time%airfare%aircraft%ground_service%airport%airline%distance%abbreviation%ground_fare%quantity%city%flight_no%capacity%flight+airfare%meal%restriction%airline+flight_no%ground_service+ground_fare%airfare+flight_time%cheapest%aircraft+flight+flight_no'}) (input_keys={'customer_message', 'intent_labels'}),
 Example({'customer_message': "what is american 's schedule of morning flights to atlanta", 'answer': 'flight_time', 'intent

# Define LabeledFewShot Optimizer
LabeledFewShot is the simplest optimizer. Its compile method injects samples intro the prompt.
There is not optimization going on.


In [10]:
from dspy.teleprompt import LabeledFewShot

few_shot_demos = random.sample(train_examples, k=10)
labeled_fewshot_optimizer = LabeledFewShot(k=len(few_shot_demos))
few_shot_model = labeled_fewshot_optimizer.compile(student=cot_predictor, trainset=few_shot_demos)


In [11]:
few_shot_demos

[Example({'customer_message': 'show me the lowest fare for a round trip flight from baltimore to dallas', 'answer': 'airfare', 'intent_labels': 'flight%flight_time%airfare%aircraft%ground_service%airport%airline%distance%abbreviation%ground_fare%quantity%city%flight_no%capacity%flight+airfare%meal%restriction%airline+flight_no%ground_service+ground_fare%airfare+flight_time%cheapest%aircraft+flight+flight_no'}) (input_keys={'customer_message', 'intent_labels'}),
 Example({'customer_message': "what 's the capacity of a 733", 'answer': 'capacity', 'intent_labels': 'flight%flight_time%airfare%aircraft%ground_service%airport%airline%distance%abbreviation%ground_fare%quantity%city%flight_no%capacity%flight+airfare%meal%restriction%airline+flight_no%ground_service+ground_fare%airfare+flight_time%cheapest%aircraft+flight+flight_no'}) (input_keys={'customer_message', 'intent_labels'}),
 Example({'customer_message': 'show me all meals on flights from atlanta to washington', 'answer': 'meal', 'in

### What is happenning under the hood?
### LabeledFewShot randomly selects labels

### DSPy SOURCE CODE: https://github.com/stanfordnlp/dspy/blob/793530c65a0e1721997dac0d2636f0f70ad649b6/dspy/teleprompt/vanilla.py#L6

class LabeledFewShot(Teleprompter):
    def __init__(self, k=16):
        self.k = k

    def compile(self, student, *, trainset, sample=True):
        self.student = student.reset_copy()
        self.trainset = trainset

        if len(self.trainset) == 0:
            return self.student

        rng = random.Random(0)

        for predictor in self.student.predictors():
            if sample:
                predictor.demos = rng.sample(self.trainset, min(self.k, len(self.trainset)))
            else:
                predictor.demos = self.trainset[: min(self.k, len(self.trainset))]

        return self.student

### My own summary of the implementation
DSPy samples randomly a portion of the samples as examples for in-context learning. 
There's no actual optimization process.

### How does the prompt looks like?

In [12]:
example = test_examples[0]
# without inputs(), we won't inject the inputs of the example
pred = few_shot_model(**example.inputs())
# Produce a prediction from our `cot` module, using the `example` above as input.
print("Prediction:", pred)
# Note: inspect_history() is specific to OpenAI. For Ollama, you can print the prediction directly.

Prediction: Prediction(
    reasoning='The customer is looking for a nonstop flight from St. Petersburg to Charlotte that leaves in the afternoon and arrives as soon after 5 pm as possible.',
    answer='flight+airfare'
)


## Define BootstrapFewShot Optimizer
This family of optimizers is focused on optimizing the few shot examples. Let's take an example of a Sample pipeline and see how we can use this optimizer to optimize it. From: https://dspy.ai/deep-dive/optimizers/bootstrap-fewshot/

In [13]:
from dspy.evaluate import answer_exact_match as metric
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=metric, #grading rules
    max_bootstrapped_demos=10, #number of examples to be generated by the model and added to the training set in each round Rationale + answer
    max_labeled_demos=10, #the limit for unaltered training examples that can be added to the training set in each round
    max_rounds=10, #If synthetic generation of few shot examples does not lead to improvement, the optimization process will stop after 10 rounds
)

### Optimize

In [14]:
# documentation is wrong - there is not valset: https://dspy.ai/deep-dive/optimizers/bootstrap-fewshot/
cot_few_shot_optimized = optimizer.compile(cot_predictor, trainset=train_examples)


 33%|███▎      | 12/36 [04:56<09:52, 24.68s/it]

Bootstrapped 10 full traces after 12 examples for up to 10 rounds, amounting to 36 attempts.


## Peek under the hood of DSPy source code for BootStrapFewShot training

### DSPy source code for training
class BootstrapFewShot()
    def _train(self):
        rng = random.Random(0)
        raw_demos = self.validation

        for name, predictor in self.student.named_predictors():
            augmented_demos = self.name2traces[name][: self.max_bootstrapped_demos]

            sample_size = min(self.max_labeled_demos - len(augmented_demos), len(raw_demos))
            sample_size = max(0, sample_size)

            raw_demos = rng.sample(raw_demos, sample_size)

            if dspy.settings.release >= 20230928:
                predictor.demos = raw_demos + augmented_demos
            else:
                predictor.demos = augmented_demos + raw_demos

        return self.student

I consulted with ChatGPT about this method. Source code: https://github.com/stanfordnlp/dspy/blob/main/dspy/teleprompt/bootstrap.py

_train() Purpose
Once _bootstrap() has collected and validated a set of bootstrapped demos, _train() takes over to:

Compile Final Demos for Predictors: _train() assembles the demos (both bootstrapped and labeled) for each predictor within the student model. For each predictor, it selects a mix of bootstrapped demos (from _bootstrap()) and labeled examples (raw demos from the validation set) to create a final demo set.
Random Sampling: The method performs a random sample from the raw labeled demos, ensuring the demos meet the configuration limits, such as max_labeled_demos.
Set Demos for Each Predictor: Finally, _train() updates each predictor in the student model with this finalized set of demos, effectively preparing it for use.
In essence, _bootstrap() is responsible for creating and validating bootstrapped demos, while _train() assembles a balanced set of these demos and labeled examples to finalize the student model’s training.

### My Own Summary
BootstrapFewShot has two main properties:
1. Enable you to generate additional examples
2. DSPy tests which predictions pass the validation and keep only those

## Define BootstrapFewShotWithRandomSearch

In [15]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

optimizer = BootstrapFewShotWithRandomSearch(
    metric=metric, 
    max_bootstrapped_demos=10, 
    max_labeled_demos=10,
    num_threads=10,
    num_candidate_programs=5
)

Going to sample between 1 and 10 traces per predictor.
Will attempt to bootstrap 5 candidate sets.


In [16]:
# THE CORRECT LINE:
cot_few_shot_rs_optimized = optimizer.compile(
    student=cot_predictor, 
    trainset=train_examples, 
    valset=dev_examples 
)

Average Metric: 1.00 / 1 (100.0%):   0%|          | 0/45 [00:00<?, ?it/s]

Average Metric: 16.00 / 45 (35.6%): 100%|██████████| 45/45 [03:04<00:00,  4.10s/it]

2026/06/05 21:23:36 INFO dspy.evaluate.evaluate: Average Metric: 16 / 45 (35.6%)



New best score: 35.56 for seed -3
Scores so far: [35.56]
Best score so far: 35.56
Average Metric: 32.00 / 45 (71.1%): 100%|██████████| 45/45 [05:21<00:00,  7.15s/it]

2026/06/05 21:28:58 INFO dspy.evaluate.evaluate: Average Metric: 32 / 45 (71.1%)


...


Scores so far: [35.56, 71.11, 53.33, 55.56, 46.67, 53.33, 48.89]
Best score so far: 71.11


 19%|█▉        | 7/36 [00:36<02:30,  5.19s/it]


Bootstrapped 4 full traces after 7 examples for up to 1 rounds, amounting to 7 attempts.
Average Metric: 29.00 / 45 (64.4%): 100%|██████████| 45/45 [02:23<00:00,  3.19s/it]

2026/06/05 21:52:39 INFO dspy.evaluate.evaluate: Average Metric: 29 / 45 (64.4%)



Scores so far: [35.56, 71.11, 53.33, 55.56, 46.67, 53.33, 48.89, 64.44]
Best score so far: 71.11
8 candidate programs found.


## Peek under the hood of the source code Implementation

### Source code
From: https://github.com/stanfordnlp/dspy/blob/main/dspy/teleprompt/random_search.py

       assert seed >= 0, seed

        random.Random(seed).shuffle(trainset_copy)
        size = random.Random(seed).randint(self.min_num_samples, self.max_num_samples)

        optimizer = BootstrapFewShot(
            metric=self.metric,
            metric_threshold=self.metric_threshold,
            max_bootstrapped_demos=size,
            max_labeled_demos=self.max_labeled_demos,
            teacher_settings=self.teacher_settings,
            max_rounds=self.max_rounds,
            max_errors=self.max_errors,
        )

        program = optimizer.compile(student, teacher=teacher, trainset=trainset_copy)

    evaluate = Evaluate(
        devset=self.valset,
        metric=self.metric,
        num_threads=self.num_threads,
        max_errors=self.max_errors,
        display_table=False,
        display_progress=True,
    )

    score, subscores = evaluate(program, return_all_scores=True)

    all_subscores.append(subscores)

### My own summary
Given the number of programs we will generate each time a different seed and run BootStrapFewShot with that 

# Evaluation

## Single Evaluation

In [17]:
from dspy.evaluate import answer_exact_match

# Instantiate the metric.
metric = answer_exact_match

example = test_examples[0]
# Produce a prediction from our `cot` module, using the `example` above as input.
print(example)
pred = cot_predictor(**example.inputs())
print(pred)

# Compute the metric score for the prediction.
score = metric(example, pred)

print(f"Customer message: \t {example.customer_message}\n")
print(f"Gold Response: \t {example.answer}\n")
print(f"Predicted Response: \t {pred.answer}\n")
print(f"Exact match score: {score:.2f}")

Example({'customer_message': 'find a flight between st. petersburg and charlotte the flight should leave in the afternoon and arrive as soon after 5 pm as possible it should be a nonstop flight', 'answer': 'flight', 'intent_labels': 'flight%flight_time%airfare%aircraft%ground_service%airport%airline%distance%abbreviation%ground_fare%quantity%city%flight_no%capacity%flight+airfare%meal%restriction%airline+flight_no%ground_service+ground_fare%airfare+flight_time%cheapest%aircraft+flight+flight_no'}) (input_keys={'customer_message', 'intent_labels'})
Prediction(
    reasoning='The customer message requires finding a nonstop flight from St. Petersburg to Charlotte that departs in the afternoon, arrives as soon after 5 pm as possible, and has the earliest arrival time.',
    answer='flight%cheapest%'
)
Customer message: 	 find a flight between st. petersburg and charlotte the flight should leave in the afternoon and arrive as soon after 5 pm as possible it should be a nonstop flight

Gold R

## Setup Evaluation

In [18]:
from dspy.evaluate.evaluate import Evaluate
# Set up the `evaluate_atis` function. We'll use this many times below.
print(len(train_examples))
evaluate_atis = Evaluate(devset=test_examples, num_threads=8, display_progress=True, display_table=5, provide_traceback=True)

36


## Evaluate zero shot CoT 

In [19]:
# Evaluate the program with the `answer_exact_match` metric.
# Launch evaluation.
evaluate_atis(cot_predictor, metric=metric)


Average Metric: 11.00 / 45 (24.4%): 100%|██████████| 45/45 [01:26<00:00,  1.92s/it]

2026/06/05 21:54:09 INFO dspy.evaluate.evaluate: Average Metric: 11 / 45 (24.4%)


,customer_message,example_answer,intent_labels,reasoning,pred_answer,answer_exact_match
0,find a flight between st. petersburg and charlotte the flight shou...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer message requires finding a nonstop flight from St. Pe...,flight%cheapest%,✔️ [False]
1,get flights from detroit to toronto,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"The customer message ""get flights from detroit to toronto"" is rela...",flight%airfare%city%airline%distance%\n\n[[ ## completed ## ]),✔️ [False]
2,what flights go from phoenix to salt lake city,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer message is asking about flights from Phoenix to Salt ...,flight%airfare%city%flight_no%,✔️ [False]
3,list all flights arriving in denver between 8 and 9 pm,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer is requesting information about flights arriving in D...,flight%aircraft%city%airport%airline%distance%abbreviation%,✔️ [False]
4,can you find me a flight from salt lake city to new york city next...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer message can be classified as a query for booking a fl...,flight%airline%cheapest,✔️ [False]


EvaluationResult(score=24.44, results=<list of 45 results>)

## Evaluate few shot CoT

In [20]:
evaluate_atis(few_shot_model, metric=metric)

Average Metric: 34.00 / 45 (75.6%): 100%|██████████| 45/45 [01:58<00:00,  2.64s/it]

2026/06/05 21:56:08 INFO dspy.evaluate.evaluate: Average Metric: 34 / 45 (75.6%)


,customer_message,example_answer,intent_labels,reasoning,pred_answer,answer_exact_match
0,find a flight between st. petersburg and charlotte the flight shou...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer is looking for a nonstop flight from St. Petersburg t...,flight+airfare,✔️ [False]
1,get flights from detroit to toronto,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,Not supplied for this particular example.,flight,✔️ [True]
2,what flights go from phoenix to salt lake city,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,Not supplied for this particular example.,flight,✔️ [True]
3,list all flights arriving in denver between 8 and 9 pm,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"The customer message is asking for a specific flight schedule, so ...",flight_no,✔️ [False]
4,can you find me a flight from salt lake city to new york city next...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer message is specifying a very specific flight schedule...,flight,✔️ [True]


EvaluationResult(score=75.56, results=<list of 45 results>)

## Evaluate BootstrapedFewShot

In [21]:
evaluate_atis(cot_few_shot_optimized, metric=metric)

Average Metric: 32.00 / 45 (71.1%): 100%|██████████| 45/45 [02:40<00:00,  3.57s/it]

2026/06/05 21:58:49 INFO dspy.evaluate.evaluate: Average Metric: 32 / 45 (71.1%)


,customer_message,example_answer,intent_labels,reasoning,pred_answer,answer_exact_match
0,find a flight between st. petersburg and charlotte the flight shou...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,To find a flight between St. Petersburg and Charlotte that meets t...,airfare,✔️ [False]
1,get flights from detroit to toronto,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer message is asking for flights from Detroit to Toronto...,flight,✔️ [True]
2,what flights go from phoenix to salt lake city,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"Not supplied for this particular example. However, to provide a he...",flight,✔️ [True]
3,list all flights arriving in denver between 8 and 9 pm,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,To generate a list of flights arriving in Denver between 8 and 9 p...,flight_time,✔️ [False]
4,can you find me a flight from salt lake city to new york city next...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,To find a flight from Salt Lake City to New York City next Saturda...,flight,✔️ [True]


EvaluationResult(score=71.11, results=<list of 45 results>)

## Evaluate Boostraped Random Search

In [22]:
evaluate_atis(cot_few_shot_rs_optimized, metric=metric)

Average Metric: 29.00 / 45 (64.4%): 100%|██████████| 45/45 [02:14<00:00,  2.99s/it]

2026/06/05 22:01:03 INFO dspy.evaluate.evaluate: Average Metric: 29 / 45 (64.4%)


,customer_message,example_answer,intent_labels,reasoning,pred_answer,answer_exact_match
0,find a flight between st. petersburg and charlotte the flight shou...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer is looking for a nonstop flight from St. Petersburg t...,flight_no,✔️ [False]
1,get flights from detroit to toronto,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,Not supplied for this particular example.,flight_no,✔️ [False]
2,what flights go from phoenix to salt lake city,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer is asking for a list of flights from Phoenix to Salt ...,flight_no,✔️ [False]
3,list all flights arriving in denver between 8 and 9 pm,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,Not supplied for this particular example.,flight_no,✔️ [False]
4,can you find me a flight from salt lake city to new york city next...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"The customer is looking for a specific flight on a specific date, ...",airfare,✔️ [False]


EvaluationResult(score=64.44, results=<list of 45 results>)

# Save / Load models

In [23]:
cot_predictor.save("cot_zero_shot.json")
few_shot_model.save("cot_few_shot.json")
cot_few_shot_optimized.save("cot_boostraped_few_shot.json")
cot_few_shot_rs_optimized.save("cot_bootstraped_rs_few_shot.json")

In [24]:
cot_predictor.load("cot_zero_shot.json")
few_shot_model.load("cot_few_shot.json")
cot_few_shot_optimized.load("cot_boostraped_few_shot.json")
cot_few_shot_rs_optimized.load("cot_bootstraped_rs_few_shot.json")

In [25]:
import dspy
from dspy.teleprompt import MIPROv2

# 1. Initialize the MIPROv2 optimizer
mipro_optimizer = MIPROv2(
    metric=metric,
    num_threads=10,
    auto="light" 
)

# 2. Compile your program 
few_shot_model = mipro_optimizer.compile(
    student=cot_predictor, 
    trainset=train_examples,
    valset=dev_examples, # Updated to use your 'test_examples' dataset
    requires_permission_to_run=False
)

2026/06/05 22:01:03 WARNING dspy.teleprompt.mipro_optimizer_v2: 'requires_permission_to_run' is deprecated and will be removed in a future version.
2026/06/05 22:01:03 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 45

2026/06/05 22:01:03 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/06/05 22:01:03 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/06/05 22:01:03 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


 19%|█▉        | 7/36 [00:20<01:23,  2.87s/it]


Bootstrapped 4 full traces after 7 examples for up to 1 rounds, amounting to 7 attempts.
Bootstrapping set 4/6


  3%|▎         | 1/36 [00:05<03:24,  5.83s/it]


...

2026/06/05 22:29:02 INFO dspy.evaluate.evaluate: Average Metric: 23 / 45 (51.1%)
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 51.11 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 4'].
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [35.56, 48.89, 11.11, 48.89, 55.56, 48.89, 11.11, 15.56, 51.11]
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 55.56
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: ========================


2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 10 / 10 =====



Average Metric: 7.00 / 45 (15.6%): 100%|██████████| 45/45 [00:00<00:00, 527.25it/s]

2026/06/05 22:29:02 INFO dspy.evaluate.evaluate: Average Metric: 7 / 45 (15.6%)
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 15.56 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 5'].
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [35.56, 48.89, 11.11, 48.89, 55.56, 48.89, 11.11, 15.56, 51.11, 15.56]
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 55.56
2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/06/05 22:29:02 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 11 / 10 =====



Average Metric: 25.00 / 45 (55.6%): 100%|██████████| 45/45 [00:00<00:00, 596.80it/s]

2026/06/05 22:29:03 INFO dspy.evaluate.evaluate: Average Metric: 25 / 45 (55.6%)
2026/06/05 22:29:03 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 55.56 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2026/06/05 22:29:03 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [35.56, 48.89, 11.11, 48.89, 55.56, 48.89, 11.11, 15.56, 51.11, 15.56, 55.56]
2026/06/05 22:29:03 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 55.56
2026/06/05 22:29:03 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/06/05 22:29:03 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 55.56!


In [26]:
evaluate_atis(few_shot_model, metric=metric)

Average Metric: 24.00 / 45 (53.3%): 100%|██████████| 45/45 [02:57<00:00,  3.94s/it]

2026/06/05 22:32:00 INFO dspy.evaluate.evaluate: Average Metric: 24 / 45 (53.3%)


,customer_message,example_answer,intent_labels,reasoning,pred_answer,answer_exact_match
0,find a flight between st. petersburg and charlotte the flight shou...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"To find a suitable flight, we need to look for a non-stop afternoo...",airfare,✔️ [False]
1,get flights from detroit to toronto,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer is requesting a general query for flights from Detroi...,flight,✔️ [True]
2,what flights go from phoenix to salt lake city,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer is asking about general flights from Phoenix to Salt ...,flight,✔️ [True]
3,list all flights arriving in denver between 8 and 9 pm,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"The customer is asking for a specific flight schedule, requiring m...",flight schedule or flight times,✔️ [False]
4,can you find me a flight from salt lake city to new york city next...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The customer is asking for a specific flight from Salt Lake City t...,airfare,✔️ [False]


EvaluationResult(score=53.33, results=<list of 45 results>)

In [27]:
few_shot_model.save("MiproV2Prompt.json")

In [ ]:
import dspy
import os

# 1. Define the Reflection Model for GEPA
# This model acts as the teacher to diagnose errors and rewrite your prompt
reflection_model = dspy.LM("openai/gpt-5") 

os.environ["OPENAI_API_KEY"] = "USE-YOUR-OWN-KEY" # Set your OpenAI API key as an environment variable for authentication
# 2. Define the Metric Function EXACTLY as GEPA expects it
def my_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Evaluates if the predicted intent matches the actual ground truth intent.
    Accepts all 5 arguments required by the GEPA optimizer.
    """
    # Safely extract the prediction string
    predicted_intent = pred.answer.strip().lower() if hasattr(pred, 'answer') else ""
    
    # Safely extract the ground truth string
    ground_truth_intent = gold.answer.strip().lower()
    
    # Return a boolean score (True if match, False otherwise)
    return predicted_intent == ground_truth_intent

# 3. Initialize the GEPA Optimizer
gepa_optimizer = dspy.GEPA(
    metric=my_metric, 
    reflection_lm=reflection_model, 
    num_threads=10,
    auto="light" #Light, Medium, and Heavy modes control the aggressiveness of the optimizer's mutations. Light mode makes smaller, more conservative changes to the prompt, while Heavy mode makes larger, more radical changes. Medium mode strikes a balance between the two.
)

# 4. Compile and Optimize your Program
print("Starting GEPA Optimization...")

optimized_model = gepa_optimizer.compile(
    student=cot_predictor, 
    trainset=train_examples,
    valset=dev_examples
)

print("Optimization Complete!")

2026/06/06 04:54:48 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 560 metric calls of the program. This amounts to 6.91 full evals on the train+val set.
2026/06/06 04:54:49 INFO dspy.teleprompt.gepa.gepa: Using 45 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


Starting GEPA Optimization...


GEPA Optimization:   0%|          | 0/560 [00:00<?, ?rollouts/s]2026/06/06 04:54:49 INFO dspy.evaluate.evaluate: Average Metric: 13 / 45 (28.9%)
2026/06/06 04:54:49 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.28888888888888886 over 45 / 45 examples
GEPA Optimization:   8%|▊         | 45/560 [00:00<00:01, 333.53rollouts/s]2026/06/06 04:54:49 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.28888888888888886


Average Metric: 0.00 / 3 (0.0%): 100%|██████████| 3/3 [00:00<00:00, 239.79it/s]

2026/06/06 04:54:49 INFO dspy.evaluate.evaluate: Average Metric: 0 / 3 (0.0%)


...

2026/06/06 05:39:10 INFO dspy.teleprompt.gepa.gepa: Iteration 12: Proposed new text for predict: Task: Classify a single natural-language air travel query into exactly one intent label.

Inputs you will receive:
- customer_message: a free-form user query about air travel.
- intent_labels: a single line containing all valid labels for this task, delimited by %.

Your output:
- Return exactly one label string that exactly matches one of the labels in intent_labels (case and punctuation must match).
- Output only the label on a single line. Do not include explanations, reasoning, extra words, punctuation, or additional whitespace.

Core decision strategy:
1) Identify the specific information the user wants returned (the requested output), not the filters/constraints (cities, dates, times, airlines usually act as constraints).
2) Apply high-priority cue rules (e.g., “how many” → quantity, “cheapest” → cheapest, “what does/what is [code]” → abbreviation).
3) If the query clearly requests tw

Average Metric: 1.00 / 3 (33.3%): 100%|██████████| 3/3 [00:14<00:00,  4.78s/it]

2026/06/06 05:42:31 INFO dspy.evaluate.evaluate: Average Metric: 1 / 3 (33.3%)


2026/06/06 05:43:16 INFO dspy.teleprompt.gepa.gepa: Iteration 13: Proposed new text for predict: Task: Classify a single natural-language air travel query into exactly one intent label.

Inputs you will receive:
- customer_message: a free-form user query about air travel.
- intent_labels: a single line containing all valid labels for this task, delimited by %.

Your output:
- Return exactly one label string that exactly matches one of the labels in intent_labels (case and punctuation must match).
- Output only the label on a single line. Do not include explanations, reasoning, extra words, punctuation, or additional whitespace.

Core decision strategy:
1) Identify the specific information the user wants returned (the requested output), not the filters/constraints (cities, dates, times, airlines usually act as constraints).
2) Apply high-priority cue rules (e.g., “how many” → quantity, “cheapest” → cheapest, “what does/what is [code]” → abbreviation).
3) If the query clearly requests tw

Optimization Complete!


In [36]:
# 1. Redefine the evaluator to guarantee it uses the fresh metric
evaluate_atis = dspy.Evaluate(
    devset=test_examples, 
    metric=my_metric, # Explicitly bind the new metric here
    num_threads=8, 
    display_progress=True, 
    display_table=5, 
    provide_traceback=True
)

# 2. Test your newly optimized model!
# (Assuming you named the output of the GEPA compilation 'optimized_model')
evaluation_score = evaluate_atis(optimized_model)

print(f"Optimized Model Score: {evaluation_score}")

Average Metric: 33.00 / 45 (73.3%): 100%|██████████| 45/45 [03:09<00:00,  4.21s/it]

2026/06/06 05:49:15 INFO dspy.evaluate.evaluate: Average Metric: 33 / 45 (73.3%)


,customer_message,example_answer,intent_labels,reasoning,pred_answer,my_metric
0,find a flight between st. petersburg and charlotte the flight shou...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,The query is asking to find a nonstop flight from St. Petersburg t...,flight+airfare%,✔️ [False]
1,get flights from detroit to toronto,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"The user is requesting a flight query, specifically asking for fli...",flight,✔️ [True]
2,what flights go from phoenix to salt lake city,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"For this query, we can extract that the user wants to know about f...",flight,✔️ [True]
3,list all flights arriving in denver between 8 and 9 pm,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"Based on the customer message ""list all flights arriving in Denver...",airfare+flight_time,✔️ [False]
4,can you find me a flight from salt lake city to new york city next...,flight,flight%flight_time%airfare%aircraft%ground_service%airport%airline...,"The query contains specific time constraints (""next saturday"", ""be...",flight_time,✔️ [False]


Optimized Model Score: EvaluationResult(score=73.33, results=<list of 45 results>)


In [37]:
optimized_model.save("GEPAPrompt.json")

In [ ]:
cot_predictor.parameters()

[Predict(StringSignature(customer_message, intent_labels -> reasoning, answer
     instructions='Classify the customer message into one of the intent labels.\nThe output should be only the predicted class as a single intent label.'
     customer_message = Field(annotation=str required=True json_schema_extra={'desc': 'Customer message during customer service interaction', '__dspy_field_type': 'input', 'IS_TYPE_UNDEFINED': True, 'prefix': 'Customer Message:'})
     intent_labels = Field(annotation=str required=True json_schema_extra={'desc': 'Labels that represent customer intent', '__dspy_field_type': 'input', 'IS_TYPE_UNDEFINED': True, 'prefix': 'Intent Labels:'})
     reasoning = Field(annotation=str required=True json_schema_extra={'desc': '${reasoning}', '__dspy_field_type': 'output', 'prefix': 'Reasoning:'})
     answer = Field(annotation=str required=True json_schema_extra={'desc': "a label best matching customer's intent ", '__dspy_field_type': 'output', 'IS_TYPE_UNDEFINED': True

In [ ]:
few_shot_model.demos

In [ ]:
#cot_few_shot_optimized.demos

In [ ]:
#cot_few_shot_rs_optimized.demos

# Remove bootstrapping

In [ ]:
from dspy.evaluate import answer_exact_match as metric
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=metric,
    max_bootstrapped_demos=4, # no need for boostrapped
    max_labeled_demos=40, # increase examples
    max_rounds=20,
)

# documentation is wrong - there is not valset: https://dspy.ai/deep-dive/optimizers/bootstrap-fewshot/
cot_few_shot_to_bootstrap = optimizer.compile(cot_predictor, trainset=train_examples)


evaluate_atis(cot_few_shot_to_bootstrap, metric=metric)
# bootrsapped_demos: generated demos?
# max_labeled_demos: input demos